# DBTS TRAIN-Only Diagnostic

This notebook runs the model strictly on the TRAIN window only. It does not use the validation/test/Volishan sets for fitting or scoring.

It reports:
- Sharpe
- Sortino
- max drawdown
- win rate
- number of trades
- long / short / flat counts
- target switches
- selected target distribution
- sector summary
- full trade log
- confusion matrix and classification report on TRAIN only
- equity curve and drawdown plot
- cumulative PnL by sector and by target
- target selection charts

Important: these are in-sample TRAIN metrics, so they will usually look better than any held-out result. They are useful for diagnostics, not for estimating real generalization.

In [ ]:
from __future__ import annotations

import json
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from xgboost import XGBClassifier

from config import SECTORS
from strategy.backtester import Backtester
from strategy.pipeline import StrategyPipeline
from strategy.position_manager import PositionManager, summarize_completed_trades
from strategy.splits import chrono_split
from strategy.strategy_config import StrategyConfig

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams.update({
    "figure.figsize": (14, 7),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titleweight": "bold",
})


@dataclass
class TrainOnlyFold:
    retrain_date: pd.Timestamp
    train_idx: pd.DatetimeIndex
    predict_idx: pd.DatetimeIndex


def load_train_only_folds(md, cfg: StrategyConfig, split) -> list[TrainOnlyFold]:
    diag_path = Path("outputs/train_only_deep_diagnostic.json")
    if not diag_path.exists():
        raise FileNotFoundError("outputs/train_only_deep_diagnostic.json is required for the train-only retrain timeline")

    diag = json.loads(diag_path.read_text(encoding="utf-8"))
    retrain_dates = sorted({pd.Timestamp(row["retrain_date"]) for row in diag.get("shadow_selected", [])})
    if not retrain_dates:
        raise ValueError("No retrain dates found in outputs/train_only_deep_diagnostic.json")

    index = pd.DatetimeIndex(md.prices.index).sort_values()
    folds: list[TrainOnlyFold] = []
    for i, retrain_date in enumerate(retrain_dates):
        next_retrain = retrain_dates[i + 1] if i + 1 < len(retrain_dates) else split.train_end
        train_idx = index[index < retrain_date]
        predict_idx = index[(index >= retrain_date) & (index < next_retrain) & (index < split.train_end)]
        if len(train_idx) and len(predict_idx):
            folds.append(TrainOnlyFold(retrain_date=retrain_date, train_idx=train_idx, predict_idx=predict_idx))
    return folds


def train_only_feature_columns(panel: pd.DataFrame) -> list[str]:
    excluded = {
        "date", "etf", "sector", "target", "predictors", "target_price",
        "shadow_price", "next_ret", "label", "spread_signal",
        "ann_vol", "residual_z", "price_residual", "residual_ewm_mean",
        "residual_ewm_std", "residual_roll_mean", "residual_roll_std",
        "fold_role", "retrain_date", "role", "true_label",
    }
    feature_cols = [c for c in panel.columns if c not in excluded]
    return [c for c in feature_cols if pd.api.types.is_numeric_dtype(panel[c])]


def build_position_manager(cfg: StrategyConfig) -> PositionManager:
    return PositionManager(
        long_entry_confidence=float(cfg.pm_entry_confidence),
        short_entry_confidence=float(cfg.pm_entry_confidence),
        flat_probability_block=float(cfg.flat_probability_block),
        entry_residual_threshold=float(cfg.pm_entry_residual_z),
        mean_reversion_exit=float(cfg.pm_exit_residual_z),
        opposite_signal_confidence=float(cfg.pm_opposite_confidence),
        stop_loss=float(cfg.pm_stop_loss),
        take_profit=float(cfg.pm_take_profit),
        max_holding_days=int(cfg.pm_max_holding_days),
        allow_flip=bool(cfg.pm_allow_flip),
    )


def fit_train_only_classifier(train_panel: pd.DataFrame, cfg: StrategyConfig):
    data = train_panel.copy().dropna(subset=["label", "next_ret"])
    feature_cols = train_only_feature_columns(data)
    if not feature_cols:
        raise ValueError("No numeric feature columns found for the train-only classifier")

    X = data[feature_cols].apply(pd.to_numeric, errors="coerce")
    X = X.groupby(data["target"]).ffill().fillna(0.0)
    y = data["label"].astype(int)
    y_idx = y.map({-1: 0, 0: 1, 1: 2}).astype(int)
    if set(y_idx.unique()) != {0, 1, 2}:
        raise ValueError("TRAIN-only classifier needs all three labels (-1, 0, 1) present in the training panel")

    params = dict(cfg.clf_params)
    params.update({
        "num_class": 3,
        "objective": "multi:softprob",
        "eval_metric": "mlogloss",
        "random_state": int(cfg.random_state),
        "verbosity": 0,
    })
    model = XGBClassifier(**params)
    model.fit(X, y_idx)

    proba_raw = pd.DataFrame(model.predict_proba(X), index=X.index, columns=[0, 1, 2])
    proba = pd.DataFrame(index=X.index)
    proba["P_short"] = proba_raw[0]
    proba["P_flat"] = proba_raw[1]
    proba["P_long"] = proba_raw[2]

    signal = pd.Series(model.predict(X), index=X.index).map({0: -1, 1: 0, 2: 1}).astype(int)

    scored = data.copy()
    scored["true_label"] = scored["label"].astype(int)
    scored["signal"] = signal
    scored = pd.concat([scored, proba], axis=1)
    return scored, model, feature_cols


def sortino_ratio(returns: pd.Series) -> float:
    r = pd.Series(returns).dropna().astype(float)
    if r.empty:
        return float("nan")
    downside = r[r < 0]
    downside_std = float(downside.std(ddof=0)) if len(downside) else float("nan")
    if not np.isfinite(downside_std) or downside_std == 0:
        return float("nan")
    return float((r.mean() / downside_std) * np.sqrt(252))

In [ ]:
cfg = StrategyConfig(force_recompute=True, make_plots=False)
pipeline = StrategyPipeline(cfg)
market_data = pipeline.load_data()
split = chrono_split(market_data.prices.index, cfg)
folds = load_train_only_folds(market_data, cfg, split)

print(f"TRAIN window: {split.train_idx[0].date()} -> {split.train_end.date()}")
print(f"Train-only retrain folds: {len(folds)}")

NameError: name 'StrategyConfig' is not defined

## How to read the result

If this looks strong on TRAIN, that is expected and not proof the model is useful out of sample. A model can look very good on the data it was trained on while still failing on new dates.

For this notebook, the right question is not "is the TRAIN score high?" but "what exactly is the model doing, and does it overfit badly enough that it would collapse on held-out data?"